# High-rate stairway codes — logical error rate vs p

Curves of logical error rate vs physical error rate `p` for each `[[n,k,d]]` stairway code in the
`threshold_stairway` collection (decoded with the Tesseract decoder)."""

In [ ]:
import sys
import datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt

# Shared backend (also used by tools/plot_high_rate.py) so the notebook and the
# CLI produce identical figures.
from high_rate_analysis import (
    load_collection,
    plot_panel,
    add_pbz_axis,
    set_paper_style,
)

%load_ext autoreload
%autoreload 2

In [ ]:
CSV_GLOB = '../out/collection/threshold_stairway,*.csv'
BIAS = 10

# Load + merge + resolve [[n,k,d]] codes at the chosen Z-bias (see
# high_rate_analysis.load_collection).
df, code_order = load_collection(CSV_GLOB, BIAS)

n_zero = int((df['errors'] == 0).sum())
print(f'{len(df)} rows; bias={BIAS:g}; codes={code_order}')
print(f'zero-error rows kept: {n_zero}/{len(df)} (sinter draws them as upper bounds)')
print(f'low-confidence shots: {int(df["low_confidence_shots"].sum())} / {int(df["shots"].sum())} total')
df.head()

In [ ]:
# Single stairway panel. Unlike the two-panel high_rate.ipynb figure (which
# rescales stairway curves by d/16), this standalone view counts a per-
# observable failure (unit 1). plot_panel / add_pbz_axis are shared with
# tools/plot_high_rate.py.
set_paper_style()

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
plot_panel(ax, df, code_order, label_suffix='stairway code',
           failure_units_per_shot_func=lambda s: 1)

ax.set_xlim(float(df['p'].min()) * 0.7, float(df['p'].max()) * 1.4)
# Override plot_panel's default "residual error rate" label for this figure.
ax.set_xlabel(r'Physical error rate $p$')

fig.tight_layout()
fig.subplots_adjust(bottom=0.24)
add_pbz_axis(ax, BIAS)
fig.set_dpi(150)

out_dir = Path('../out/figures')
out_dir.mkdir(parents=True, exist_ok=True)
date = datetime.date.today().strftime('%Y%m%d')
out_path = out_dir / f'{date}_high_rate_stairway_bias{int(BIAS)}.pdf'
fig.savefig(out_path, bbox_inches='tight')
print(f'Saved {out_path}')
plt.show()